# 🐛 European Insect Classifier — Full Training Pipeline

**Upload this notebook to Kaggle and run all cells.**

What this notebook does:
1. Installs dependencies
2. Queries iNaturalist API to build a European insect manifest
3. Downloads images directly from iNaturalist S3 into Kaggle scratch space
4. Trains EfficientNetV2-S with two-phase fine-tuning
5. Evaluates on test set
6. Saves the checkpoint as a Kaggle output (downloadable)

**Runtime:** Enable GPU in `Settings → Accelerator → GPU T4 x2`  
**Expected duration:** 8-18 hours (depending on GPU and species count)  
**GPU quota used:** ~15-20 of your 30 free hours/week

---
### Configuration — edit these before running


In [ ]:
# ════════════════════════════════════════════════
#  CONFIGURATION — adjust these to your needs
# ════════════════════════════════════════════════

CFG = {
    # ── Geography ───────────────────────────────
    # Europe bounding box (lat/lon)
    'eu_bbox': {'nelat': 71, 'nelng': 45, 'swlat': 34, 'swlng': -25},

    # ── Species scope ───────────────────────────
    # 'Insecta' = all insects
    # 'Lepidoptera' = butterflies/moths only (faster, good first run)
    # 'Coleoptera' = beetles only
    'iconic_taxon': 'Insecta',

    # Min observations per species to include
    # 200 = ~800-1200 EU species, good quality
    # 500 = ~400-600 species, faster training
    'min_obs_per_species': 200,

    # Images to download per species (train+val+test combined)
    'images_per_species': 250,

    # Hard cap on total species (set None for all)
    # Start with 200 for a first full run (~50k images, ~5h training)
    'max_species': 200,

    # ── Model ───────────────────────────────────
    'model_name':      'efficientnetv2_s',
    'image_size':      224,
    'batch_size':      128,     # T4 has 16GB VRAM — 128 fits comfortably
    'epochs':          30,
    'warmup_epochs':   5,
    'lr':              3e-4,
    'weight_decay':    1e-4,
    'label_smoothing': 0.1,
    'num_workers':     4,

    # ── Splits ──────────────────────────────────
    'train_frac': 0.75,
    'val_frac':   0.15,
    # test = remaining 0.10

    # ── Paths ───────────────────────────────────
    'data_dir':   '/kaggle/working/images',
    'output_dir': '/kaggle/working/output',
}

print('Configuration loaded.')
print(f"  Scope: {CFG['iconic_taxon']} in Europe")
print(f"  Min observations/species: {CFG['min_obs_per_species']}")
print(f"  Images/species: {CFG['images_per_species']}")
print(f"  Max species: {CFG['max_species']}")
print(f"  Model: {CFG['model_name']}")

In [ ]:
# ════════════════════════════════════════════════
#  CELL 1 — Install dependencies
# ════════════════════════════════════════════════

# Kaggle already has torch, torchvision. We just need timm.
import subprocess
subprocess.run(['pip', 'install', '-q', 'timm>=1.0.0'], check=True)
print('Dependencies ready.')

In [ ]:
# ════════════════════════════════════════════════
#  CELL 2 — Imports
# ════════════════════════════════════════════════

import os, json, time, random, warnings
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.request, urllib.error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
import timm
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Path(CFG['data_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)

In [ ]:
# ════════════════════════════════════════════════
#  CELL 3 — Build species manifest from iNaturalist API
# ════════════════════════════════════════════════

INAT_API = 'https://api.inaturalist.org/v1'

def api_get(url, retries=3):
    for i in range(retries):
        try:
            req = urllib.request.Request(
                url, headers={'User-Agent': 'InsectClassifier/1.0'})
            with urllib.request.urlopen(req, timeout=20) as r:
                return json.loads(r.read())
        except Exception as e:
            if i == retries - 1: raise
            time.sleep(2 ** i)

def get_eu_species(iconic_taxon, min_obs, max_species):
    """
    Fetches species with sufficient EU observations from iNat API.
    Returns list of {taxon_id, name, obs_count} dicts.
    """
    bbox = CFG['eu_bbox']
    species = []
    page = 1
    per_page = 100

    print(f'Querying iNaturalist for {iconic_taxon} species in Europe...')
    pbar = tqdm(desc='Species fetched', unit='sp')

    while True:
        url = (
            f'{INAT_API}/observations/species_counts'
            f'?iconic_taxa={iconic_taxon}'
            f'&quality_grade=research'
            f'&photos=true'
            f'&nelat={bbox["nelat"]}&nelng={bbox["nelng"]}'
            f'&swlat={bbox["swlat"]}&swlng={bbox["swlng"]}'
            f'&per_page={per_page}&page={page}'
        )
        data = api_get(url)
        results = data.get('results', [])
        if not results:
            break

        for r in results:
            if r['count'] >= min_obs:
                species.append({
                    'taxon_id':  r['taxon']['id'],
                    'name':      r['taxon']['name'],
                    'rank':      r['taxon']['rank'],
                    'obs_count': r['count'],
                })
                pbar.update(1)

        if len(results) < per_page:
            break
        if max_species and len(species) >= max_species:
            break
        page += 1
        time.sleep(0.3)

    pbar.close()

    # Sort by observation count descending (most-observed = best data quality)
    species.sort(key=lambda x: x['obs_count'], reverse=True)
    if max_species:
        species = species[:max_species]

    return species


manifest_path = Path(CFG['output_dir']) / 'species_manifest.json'

if manifest_path.exists():
    print('Loading existing manifest...')
    SPECIES = json.load(open(manifest_path))
else:
    SPECIES = get_eu_species(
        CFG['iconic_taxon'],
        CFG['min_obs_per_species'],
        CFG['max_species']
    )
    with open(manifest_path, 'w') as f:
        json.dump(SPECIES, f, indent=2)

print(f'\nManifest: {len(SPECIES)} species')
print('Top 5 by observation count:')
for s in SPECIES[:5]:
    print(f"  {s['name']:<40} {s['obs_count']:>6} observations")

In [ ]:
# ════════════════════════════════════════════════
#  CELL 4 — Download images from iNaturalist S3
# ════════════════════════════════════════════════

def fetch_photo_urls(taxon_id, limit, bbox):
    """Returns list of (photo_id, url) for one species."""
    photos = []
    page = 1
    while len(photos) < limit:
        url = (
            f'{INAT_API}/observations'
            f'?taxon_id={taxon_id}&quality_grade=research&photos=true'
            f'&nelat={bbox["nelat"]}&nelng={bbox["nelng"]}'
            f'&swlat={bbox["swlat"]}&swlng={bbox["swlng"]}'
            f'&per_page=100&page={page}&order_by=votes'
        )
        try:
            data = api_get(url)
        except Exception:
            break
        results = data.get('results', [])
        if not results: break
        for obs in results:
            for p in obs.get('photos', []):
                med_url = p.get('url', '').replace('square', 'medium')
                if med_url:
                    photos.append((str(p['id']), med_url))
                    if len(photos) >= limit: break
            if len(photos) >= limit: break
        if len(results) < 100: break
        page += 1
        time.sleep(0.3)
    return photos[:limit]


def download_one(args):
    photo_id, url, dest = args
    if Path(dest).exists():
        return True
    try:
        ext = url.split('.')[-1].split('?')[0] or 'jpg'
        req = urllib.request.Request(
            url, headers={'User-Agent': 'InsectClassifier/1.0'})
        with urllib.request.urlopen(req, timeout=15) as r:
            data = r.read()
        with open(dest, 'wb') as f:
            f.write(data)
        return True
    except Exception:
        return False


def download_species(species_list, data_dir, images_per_species, bbox, workers=16):
    data_dir = Path(data_dir)
    total_ok = total_fail = 0

    for sp in tqdm(species_list, desc='Species'):
        tid  = sp['taxon_id']
        sdir = data_dir / str(tid)
        sdir.mkdir(exist_ok=True)

        # Skip if already have enough images
        existing = list(sdir.iterdir())
        if len(existing) >= images_per_species:
            continue

        photos = fetch_photo_urls(tid, images_per_species + 20, bbox)
        tasks  = []
        for photo_id, url in photos:
            ext  = url.split('.')[-1].split('?')[0] or 'jpg'
            dest = sdir / f'{photo_id}.{ext}'
            tasks.append((photo_id, url, str(dest)))

        ok = fail = 0
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for result in pool.map(download_one, tasks):
                if result: ok += 1
                else:      fail += 1

        # Trim to limit
        files = sorted(sdir.iterdir())
        for extra in files[images_per_species:]:
            extra.unlink()

        total_ok += ok; total_fail += fail

    total_images = sum(
        1 for _ in Path(data_dir).rglob('*') if _.is_file()
    )
    print(f'\nDownload complete: {total_ok} OK, {total_fail} failed')
    print(f'Total images on disk: {total_images:,}')


download_species(
    SPECIES,
    CFG['data_dir'],
    CFG['images_per_species'],
    CFG['eu_bbox'],
    workers=16
)

In [ ]:
# ════════════════════════════════════════════════
#  CELL 5 — Dataset & DataLoaders
# ════════════════════════════════════════════════

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_transform(image_size, mode):
    if mode == 'train':
        return transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.55, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
            transforms.RandomRotation(25),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),
        ])
    return transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])

class InsectDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform
    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        return self.transform(img), label

def build_splits(data_dir, species_list, train_frac, val_frac, min_images=50):
    id_to_name = {str(s['taxon_id']): s['name'] for s in species_list}
    class_names = []
    train_s, val_s, test_s = [], [], []

    for class_dir in sorted(Path(data_dir).iterdir()):
        if not class_dir.is_dir(): continue
        images = [
            p for p in class_dir.iterdir()
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
        ]
        if len(images) < min_images: continue

        label = len(class_names)
        class_names.append(id_to_name.get(class_dir.name, class_dir.name))
        images = sorted(images)
        n = len(images)
        t_end = int(n * train_frac)
        v_end = t_end + int(n * val_frac)
        train_s.extend([(p, label) for p in images[:t_end]])
        val_s.extend(  [(p, label) for p in images[t_end:v_end]])
        test_s.extend( [(p, label) for p in images[v_end:]])

    return train_s, val_s, test_s, class_names


train_s, val_s, test_s, CLASS_NAMES = build_splits(
    CFG['data_dir'], SPECIES, CFG['train_frac'], CFG['val_frac']
)
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes: {NUM_CLASSES}  Train: {len(train_s)}  Val: {len(val_s)}  Test: {len(test_s)}')

# Save class map
CLASS_MAP = {i: name for i, name in enumerate(CLASS_NAMES)}
with open(Path(CFG['output_dir']) / 'class_map.json', 'w') as f:
    json.dump(CLASS_MAP, f, indent=2)

sz = CFG['image_size']
train_ds = InsectDataset(train_s, make_transform(sz, 'train'))
val_ds   = InsectDataset(val_s,   make_transform(sz, 'val'))
test_ds  = InsectDataset(test_s,  make_transform(sz, 'test'))

# Weighted sampler for class balance
labels  = [s[1] for s in train_s]
counts  = Counter(labels)
weights = [1.0 / counts[l] for l in labels]
sampler = WeightedRandomSampler(weights, len(weights))

kw = dict(batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
          pin_memory=True)
train_loader = DataLoader(train_ds, sampler=sampler, **kw)
val_loader   = DataLoader(val_ds,   shuffle=False,   **kw)
test_loader  = DataLoader(test_ds,  shuffle=False,   **kw)

In [ ]:
# ════════════════════════════════════════════════
#  CELL 6 — Model, Loss, Optimizer
# ════════════════════════════════════════════════

model = timm.create_model(
    CFG['model_name'],
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=0.3,
    drop_path_rate=0.2,
).to(DEVICE)

# Use DataParallel if Kaggle gives you dual-GPU (T4 x2)
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss(
    label_smoothing=CFG['label_smoothing']
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG["model_name"]}')
print(f'Parameters: {total_params/1e6:.1f}M total, {trainable_params/1e6:.1f}M trainable')

In [ ]:
# ════════════════════════════════════════════════
#  CELL 7 — Training helpers
# ════════════════════════════════════════════════

def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = correct = total = 0
    for imgs, labels in tqdm(loader, desc='  train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        # Mixed precision (AMP) — halves VRAM use, speeds up ~30%
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = total = 0
    correct = {1: 0, 3: 0, 5: 0}
    for imgs, labels in tqdm(loader, desc='   eval', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        total      += imgs.size(0)
        for k in [1, 3, 5]:
            if k <= logits.size(1):
                topk = logits.topk(k, dim=1).indices
                correct[k] += (topk == labels.unsqueeze(1)).any(1).sum().item()
    return {'loss': total_loss/total,
            'top1': correct[1]/total,
            'top3': correct[3]/total,
            'top5': correct[5]/total}

print('Training helpers defined.')

In [ ]:
# ════════════════════════════════════════════════
#  CELL 8 — Phase 1: Warm up classifier head
# ════════════════════════════════════════════════

print(f'Phase 1: Warm up ({CFG["warmup_epochs"]} epochs, frozen backbone)')

# Freeze everything, unfreeze only the classifier head
_model = model.module if hasattr(model, 'module') else model
for p in _model.parameters():        p.requires_grad_(False)
for p in _model.classifier.parameters(): p.requires_grad_(True)

opt1    = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=CFG['weight_decay']
)
sched1  = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt1, T_max=CFG['warmup_epochs'])
scaler  = torch.cuda.amp.GradScaler()
history = []

for epoch in range(1, CFG['warmup_epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, opt1, criterion, scaler)
    val_m = evaluate(model, val_loader, criterion)
    sched1.step()
    print(f'  Ep {epoch:02d} | loss {tr_loss:.4f} | '
          f'train {tr_acc:.3f} | val top1 {val_m["top1"]:.3f} | '
          f'val top3 {val_m["top3"]:.3f} | {time.time()-t0:.0f}s')
    history.append({'phase': 1, 'epoch': epoch, **val_m})

In [ ]:
# ════════════════════════════════════════════════
#  CELL 9 — Phase 2: Full fine-tune
# ════════════════════════════════════════════════

remaining = CFG['epochs'] - CFG['warmup_epochs']
print(f'Phase 2: Full fine-tune ({remaining} epochs)')

for p in model.parameters(): p.requires_grad_(True)

opt2   = torch.optim.AdamW(
    model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2, T_max=remaining, eta_min=1e-6)

best_val  = 0.0
save_path = Path(CFG['output_dir']) / 'best_model.pth'

for epoch in range(CFG['warmup_epochs'] + 1, CFG['epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, opt2, criterion, scaler)
    val_m = evaluate(model, val_loader, criterion)
    sched2.step()

    history.append({'phase': 2, 'epoch': epoch, **val_m})

    marker = ''
    if val_m['top1'] > best_val:
        best_val = val_m['top1']
        _m = model.module if hasattr(model, 'module') else model
        torch.save({
            'epoch':       epoch,
            'model_state': _m.state_dict(),
            'model_name':  CFG['model_name'],
            'num_classes': NUM_CLASSES,
            'val_top1':    best_val,
            'class_map':   CLASS_MAP,
        }, save_path)
        marker = ' ← saved'

    print(f'  Ep {epoch:02d} | loss {tr_loss:.4f} | '
          f'train {tr_acc:.3f} | val top1 {val_m["top1"]:.3f} | '
          f'val top3 {val_m["top3"]:.3f} | {time.time()-t0:.0f}s{marker}')

# Save training history
with open(Path(CFG['output_dir']) / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# ════════════════════════════════════════════════
#  CELL 10 — Test evaluation & summary
# ════════════════════════════════════════════════

import timm as _timm

ckpt  = torch.load(save_path, map_location=DEVICE)
model_eval = _timm.create_model(
    ckpt['model_name'], pretrained=False, num_classes=ckpt['num_classes']
).to(DEVICE)
model_eval.load_state_dict(ckpt['model_state'])

test_m = evaluate(model_eval, test_loader, criterion)

print('=' * 55)
print('FINAL RESULTS')
print('=' * 55)
print(f'  Species:  {NUM_CLASSES}')
print(f'  Test Top-1: {test_m["top1"]:.1%}')
print(f'  Test Top-3: {test_m["top3"]:.1%}')
print(f'  Test Top-5: {test_m["top5"]:.1%}')
print('=' * 55)
print(f'\nCheckpoint: {save_path}')
print(f'Size: {save_path.stat().st_size / 1e6:.0f} MB')
print()
print('Download this file from Kaggle Output panel →')
print('  best_model.pth')
print('  class_map.json')
print()
print('Then run 04_export.py locally on your Mac to get .onnx and .mlpackage')